# Обучение baseline YOLOv8 — Fashionpedia (11 классов)

Первая из 5 моделей проекта. Цель: наладить пайплайн **обучение → метрики → сохранение** и получить первые реальные метрики на **test**-сплите.

Ноутбук рассчитан на **Kaggle** с включёнными **GPU** и (для установки пакетов) **Internet**.

Данные подготовлены заранее и подключены через **Add Input** (read-only). Код берётся из GitHub-репозитория.

**Порядок:** установка → клон репозитория → починка `data.yaml` → обучение → метрики (общие + per-class) → сохранение результатов → **Save Version**.

## 0. Установка и проверка Ultralytics

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

import ultralytics, torch
ultralytics.checks()
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Клонирование репозитория с кодом

При повторном запуске в той же сессии `git clone` падает с «destination path already exists», поэтому: если папка есть — `git pull`, иначе — клон.

> Замените `REPO_URL` на адрес вашего репозитория, если username в GitHub отличается.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/neuromindgpt/cv-fashion-object-detection.git"  # при необходимости замените
REPO_DIR = "/kaggle/working/cv-fashion-object-detection"

if os.path.exists(REPO_DIR):
    print("Репозиторий уже склонирован — обновляю (git pull)")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("sys.path[0]:", sys.path[0])

## 2. Починка `data.yaml` (КРИТИЧНО)

Данные подключены read-only в `/kaggle/input/...`. Внутри `data.yaml` поле `path:` указывает на устаревший `/kaggle/working/...` от сессии подготовки данных. Редактировать файл в `/kaggle/input/` нельзя.

**Решение:** копируем `data.yaml` в `/kaggle/working/`, подменяем `path:` на актуальный корень данных и проверяем, что пути к `images/{train,val,test}` разрешаются.

In [ ]:
import yaml
from pathlib import Path

# Актуальный корень подготовленных данных (Add Input, read-only).
DATA_ROOT = Path(
    "/kaggle/input/notebooks/neuromindgpt/"
    "cv-fashion-object-detection/cv-fashion-object-detection/data/processed"
)
assert DATA_ROOT.exists(), f"Не найден каталог данных: {DATA_ROOT}. Проверьте Add Input."

src_yaml = DATA_ROOT / "data.yaml"
data = yaml.safe_load(open(src_yaml, encoding="utf-8"))
data["path"] = str(DATA_ROOT)  # подмена устаревшего path

WORK_YAML = Path("/kaggle/working/data.yaml")
yaml.safe_dump(data, open(WORK_YAML, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)

print("nc:", data["nc"])
print("names:", data["names"])
for split in ("train", "val", "test"):
    d = DATA_ROOT / data[split]
    n = len(list(d.glob("*.jpg"))) if d.exists() else 0
    print(f"{split:5} -> {d}  [{'OK' if d.exists() else 'НЕ НАЙДЕНО'}, {n} изображений]")

## 3. Обучение baseline

`build_model()` создаёт YOLOv8n с предобученными весами; `train()` обучает и сразу оценивает на **test**. Для baseline берём **20 эпох** (не 50 из конфига — цель первого прогона проверить пайплайн end-to-end). Результаты Ultralytics пишутся в `/kaggle/working/results/logs/yolov8_baseline` (переживут сессию через Save Version).

In [ ]:
from src.utils.utils import load_config
from src.models.yolo import build_model
from src.training.train import train

config = load_config(Path(REPO_DIR) / "configs" / "default.yaml")
num_classes = data["nc"]

model = build_model(num_classes, config=config, data_yaml=str(WORK_YAML))

metrics = train(
    model,
    str(WORK_YAML),
    config,
    epochs=20,                 # baseline override (в конфиге для финала 50)
    split="test",             # честная итоговая оценка на отложенном test
    project="/kaggle/working/results/logs",
    name="yolov8_baseline",
)

## 4. Итоговые метрики (общие + per-class)

In [ ]:
import pandas as pd

print("=== Общие метрики (test) ===")
print(f"mAP@0.5:      {metrics['map50']:.4f}")
print(f"mAP@0.5:0.95: {metrics['map50_95']:.4f}")
print(f"Precision:    {metrics['precision']:.4f}")
print(f"Recall:       {metrics['recall']:.4f}")
print(f"F1:           {metrics['f1']:.4f}")
print(f"Изображений в test: {metrics['num_images']}")

df = (
    pd.DataFrame(metrics["per_class"]).T[["map50", "map50_95", "precision", "recall", "f1"]]
    .sort_values("map50_95", ascending=False)
)
df.round(4)

In [ ]:
# Лучшие и худшие классы — пригодятся для анализа (§3.6-3.7): связь с дисбалансом и размером объектов.
best = df.index[0]
worst = df.index[-1]
print(f"Лучший класс:  {best}  (mAP@0.5:0.95 = {df.loc[best, 'map50_95']:.4f})")
print(f"Худший класс:  {worst}  (mAP@0.5:0.95 = {df.loc[worst, 'map50_95']:.4f})")

## 5. Сохранение результатов (переживают сессию)

Метрики (JSON + CSV) и артефакты обучения (веса `best.pt`, `results.png`, PR-кривые, confusion matrix) кладём в `/kaggle/working/`. В конце обязательно **Save Version** (Kaggle → Save Version), иначе output не сохранится.

In [ ]:
import json, shutil

OUT = Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)

json.dump(metrics, open(OUT / "metrics_test.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
df.to_csv(OUT / "per_class_metrics_test.csv", encoding="utf-8")

train_dir = Path(metrics["save_dir"]).parent / "yolov8_baseline"
weights = list((train_dir / "weights").glob("*.pt")) if (train_dir / "weights").exists() else []
plots = list(train_dir.glob("*.png")) + list(Path(metrics["save_dir"]).glob("*.png"))

print("Метрики:", OUT / "metrics_test.json", "|", OUT / "per_class_metrics_test.csv")
print("Веса:   ", [str(p) for p in weights])
print("Графики:", [p.name for p in plots])
print()
print("Всё в /kaggle/working. Теперь нажмите Save Version, чтобы сохранить output.")